In [15]:
import torch
import numpy as np
from omegaconf import OmegaConf
from functools import partial
import gymnasium as gym
import matplotlib.pyplot as plt
import re
from pathlib import Path
from tqdm.notebook import tqdm

import time
import datetime
import os

import bbrl_utils
from bbrl_utils.notebook import setup_tensorboard
from bbrl.stats import WelchTTest
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl.agents.gymnasium import ParallelGymAgent, make_env
from bbrl.workspace import Workspace
from bbrl.utils.replay_buffer import ReplayBuffer

import bbrl_gymnasium

from pmind.algorithms import DQN, DDPG, TD3, OfflineTD3
from pmind.losses import dqn_compute_critic_loss, ddqn_compute_critic_loss
from pmind.training import (
    run_dqn,
    run_ddpg,
    run_td3,
)
from pmind.replay import (
    collect_policy_transitions,
    collect_uniform_transitions,
    mix_transitions,
    test_rb_compositions,
)

from pmind.visualization import plot_perf_vs_rb_composition_from_dict

from pmind.config.loader import load_config

bbrl_utils.setup()

OUTPUT_DIR = f"../results/all_performances-{datetime.datetime.now().strftime('%Y-%m-%d-%H%M%S')}"
os.mkdir(OUTPUT_DIR)

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [75]:
ENV_NAMES = (
    "CartPoleContinuous-v1",
    "Pendulum-v1",
    "MountainCarContinuous-v0",
    "LunarLanderContinuous-v3",
)
BUFFER_SIZE = 200_000
TRY_BEST = False
TRY_INTERMEDIATE = True
PROPORTIONS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
SEEDS = [0, 1, 2, 3, 4]
MAX_STEPS = 15_000
NB_EVAL_ENVS = 10
EVAL_INTERVAL = 100
BATCH_SIZE = 64
NN_ARCHITECTURE = [400,300]

In [ ]:
for ENV_NAME in ENV_NAMES[2:3]:
    print(f"Testing {ENV_NAME}:")
    cfg = load_config("td3")
    cfg = OmegaConf.create(cfg.environments[ENV_NAME])

    best_reward, _ = torch.load(
        f"../models/{ENV_NAME}/best-policy.pt", weights_only=False
    )

    # Load replay buffers from file
    rb_exploit_best = torch.load(f"../models/{ENV_NAME}/best-rb.pt", weights_only=False)
    rb_exploit_intermediate = torch.load(
        f"../models/{ENV_NAME}/intermediate-rb.pt", weights_only=False
    )
    print(rb_exploit_intermediate)

    rb_unif = torch.load(f"../models/{ENV_NAME}/unif-rb.pt", weights_only=False)

    cfg_offline = OmegaConf.create(cfg)

    cfg_offline.algorithm.n_steps = MAX_STEPS
    cfg_offline.algorithm.max_epochs = None

    cfg_offline.algorithm.batch_size = BATCH_SIZE
    cfg_offline.algorithm.architecture.actor_hidden_size = NN_ARCHITECTURE
    cfg_offline.algorithm.architecture.critic_hidden_size = NN_ARCHITECTURE
    

    cfg_offline.algorithm.eval_interval = EVAL_INTERVAL
    cfg_offline.algorithm.nb_evals = NB_EVAL_ENVS  # nb of evaluation envs in parallel

    # learning starts immediately for offline:
    cfg_offline.algorithm.learning_starts = None

    # there is no exploration in offline learning
    cfg_offline.algorithm.action_noise = None
    cfg_offline.algorithm.target_policy_noise = None

    if TRY_BEST:
        print(f"best policy ({best_reward})...")
        # Offline learning on best policy
        performances_best = test_rb_compositions(
            rb_unif,
            rb_exploit_best,
            BUFFER_SIZE,
            PROPORTIONS,
            TD3,
            cfg_offline,
            SEEDS,
        )
        torch.save(performances_best, f"{OUTPUT_DIR}/all-run-{ENV_NAME}-best-{best_reward:.0f}")
    else:
        print("skipped best policy")

    if TRY_INTERMEDIATE:
        for reward, rb in rb_exploit_intermediate[::-1]:
            print(f"intermediate policy ({reward})...")
            performances = test_rb_compositions(
                rb_unif,
                rb,
                BUFFER_SIZE,
                PROPORTIONS,
                TD3,
                cfg_offline,
                SEEDS,
            )
            torch.save(performances, f"{OUTPUT_DIR}/all-run-{ENV_NAME}-intermediate-{reward:.0f}")
    else:
        print("skipped intermediate policies")


Testing MountainCarContinuous-v0:
[(34.93184280395508, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x134044430>), (60.57007598876953, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x134044580>), (81.78364562988281, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x134045bd0>), (85.9862289428711, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x134044af0>), (89.3742904663086, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x134046f50>), (90.84980773925781, <bbrl.utils.replay_buffer.ReplayBuffer object at 0x133d87f70>)]
skipped best policy
intermediate policy (60.57007598876953)...


  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

...

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]

  0%|          | 0/15000 [00:00<?, ?it/s]